In [ ]:
!pip install opencv-python matplotlib

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


PARAMS = {
    "h_kernel_w": 80,
    "h_kernel_h": 1,
    "v_kernel_w": 1,
    "v_kernel_h": 80,
    "min_cell_area": 1000,
    "max_cell_area": 500000,
}


def robust_threshold(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    blur = cv2.GaussianBlur(gray, (5,5), 0)

    thresh = cv2.adaptiveThreshold(
        blur, 255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV,
        15, 3
    )
    return thresh




def detect_lines(thresh):
    h_kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (PARAMS["h_kernel_w"], PARAMS["h_kernel_h"])
    )

    v_kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (PARAMS["v_kernel_w"], PARAMS["v_kernel_h"])
    )

    h_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, h_kernel)
    v_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, v_kernel)

    grid = cv2.add(h_lines, v_lines)

    return h_lines, v_lines, grid



def find_cells(grid):
    contours, _ = cv2.findContours(grid, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    cells = []
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        area = w * h

        if PARAMS["min_cell_area"] < area < PARAMS["max_cell_area"]:
            cells.append((x, y, w, h))

    return cells




def visualize(img, thresh, grid, cells):
    vis = img.copy()

    for (x, y, w, h) in cells:
        cv2.rectangle(vis, (x, y), (x+w, y+h), (0,255,0), 2)

    plt.figure(figsize=(15,10))

    plt.subplot(2,2,1)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title("Original")

    plt.subplot(2,2,2)
    plt.imshow(thresh, cmap='gray')
    plt.title("Threshold")

    plt.subplot(2,2,3)
    plt.imshow(grid, cmap='gray')
    plt.title("Grid")

    plt.subplot(2,2,4)
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"Celdas detectadas: {len(cells)}")

    plt.show()


# ================================
# PIPELINE
# ================================
def run_pipeline(image_path):
    img = cv2.imread(image_path)

    thresh = robust_threshold(img)

    h_lines, v_lines, grid = detect_lines(thresh)

    cells = find_cells(grid)

    visualize(img, thresh, grid, cells)

    return cells


# ================================
# SUBIR Y EJECUTAR
# ================================
import sys, random, tempfile, os
from pathlib import Path

# ── pick one image randomly from datasets/ ───────────────────────────
NOTEBOOK_DIR = Path().resolve()
for _c in [NOTEBOOK_DIR, NOTEBOOK_DIR.parent]:
    if (_c / 'datasets').exists():
        NOTEBOOK_DIR = _c
        break
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

_exts = ('*.jpg','*.jpeg','*.png','*.bmp','*.tif','*.tiff','*.heic','*.heif','*.pdf')
_imgs = [f for p in _exts for f in (NOTEBOOK_DIR/'datasets').rglob(p)]
import src.helper as _h
from src.data.preprocessing import preprocess_image as _pp
_picked  = random.choice(_imgs)
_raw     = _h.load_image(_picked)
_pre     = _pp(_raw, target_width=1024, target_height=1448)
_img_bgr = _pre['resized']

# Use a PNG temporary file that OpenCV can reliably write
_tmp = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
_tmp.close()  # Close file so OpenCV can write to it
cv2.imwrite(_tmp.name, _img_bgr)
image_path = _tmp.name
print('Randomly picked:', _picked.name)


cells = run_pipeline(image_path)